# Image Acquisition

Acquire a full-frame image, a cropped rectangle scan, and images from multiple detectors.


### Run the servers

Make sure you are on the VPN and the AutoScript server is running. Then start the asyncroscopy Tango servers from the repository root:

```bash
uv run scripts/3_run_servers.py
```


### Imports


In [ ]:
import os
import json
import time

import tango
import numpy as np
import matplotlib.pyplot as plt
from tiled.client import from_uri

import sidpy

%matplotlib ipympl


### Ping servers


In [ ]:
DB_HOST = "10.46.217.241"
DB_PORT = 9094

os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"

server_names = ['stage', 'scan', 'eds', 'camera', 'data', 'microscope']

for name in server_names:
    device_name = f"asyncroscopy/{name}/default"
    proxy = tango.DeviceProxy(device_name)
    proxy.ping()
    print(device_name, proxy.state())


### Connect to devices


In [ ]:
scan = tango.DeviceProxy("asyncroscopy/scan/default")
microscope = tango.DeviceProxy("asyncroscopy/microscope/default")
data = tango.DeviceProxy("asyncroscopy/data/default")

# Backward-compatible aliases used by the workflow cells below.
mic_proxy = microscope
microscope_proxy = microscope

for proxy in (scan, microscope, data):
    proxy.set_timeout_millis(120_000)

print("scan      :", scan.state())
print("microscope:", microscope.state())
print("data      :", data.state())


### Start Tiled data server


In [ ]:
TILED_HOST = "10.46.217.241"
TILED_PORT = 9091
save_path = "D:/microscopedata/tiled/ahoust17/2026_05_22_AtomFab/"

data.host = TILED_HOST
data.port = TILED_PORT
data.save_path = save_path

if str(data.tiled_server).lower() != "yes":
    print("Tiled server is not responding; starting it from the DATA device...")
    config = json.loads(data.start_tiled_server())
else:
    print("Tiled server is already running.")
    config = json.loads(data.get_config())

print(json.dumps(config, indent=2))


### Configure scan


In [ ]:
scan.dwell_time 

In [ ]:
scan.dwell_time   = 1e-6   # 1 µs
scan.imsize  = 1024

print('dwell_time  :', scan.dwell_time)
print('image_width :', scan.imsize)

### Acquire a HAADF image


In [ ]:
# get_image returns DevEncoded = (json_metadata_str, raw_bytes)
json_meta, raw_bytes = mic_proxy.get_scanned_image()

metadata  = dict(json.loads(json_meta))
image = np.frombuffer(raw_bytes, dtype=metadata['dtype']).reshape(metadata['shape'])

print('Metadata:', metadata)
print('Image shape:', image.shape)
print('Image dtype:', image.dtype)

### Display the image


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, cmap='gray', interpolation='none')
ax.set_title(f"HAADF — dwell {metadata['dwell_time']*1e6:.1f} µs")
ax.axis('off')
plt.tight_layout()
plt.show()

### Build a sidpy dataset


In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

import sidpy
print('sidpy version: ', sidpy.__version__)

In [ ]:
dataset = sidpy.Dataset.from_array(image , name='HAADF')

# set dimensions
dataset.set_dimension(0, sidpy.Dimension(np.arange(image.shape[0])*.02, 'x'))
dataset.set_dimension(1, sidpy.Dimension(np.arange(image.shape[0])*.02, 'y'))


# set the dataset level plotting metadata
dataset.data_type = 'image'
dataset.units = 'counts'
dataset.quantity = 'intensity'
dataset.title = 'HAADF'

# handle one dimension of the data
dataset.set_dimension(0, sidpy.Dimension(np.arange(dataset.shape[0])*.02, 'x'))
dataset.x.dimension_type = 'spatial'
dataset.x.units = 'nm'
dataset.x.quantity = 'distance'

# handle another dimension of the data

dataset.set_dimension(1, sidpy.Dimension(np.arange(dataset.shape[1])*.02, 'y'))
dataset.y.dimension_type = 'spatial'
dataset.y.units = 'nm'
dataset.y.quantity = 'distance'

In [ ]:
dataset.metadata = metadata

In [ ]:
view = dataset.plot(scale_bar=True)

### Advanced multi-detector acquisition


In [ ]:
adv_acq_proxy = tango.DeviceProxy("tango://127.0.0.1:8890/asyncroscopy/advancedacquisition/default#dbase=no")


In [ ]:
adv_acq_proxy.state()

In [ ]:
for attr in adv_acq_proxy.get_attribute_list():
    print(f'  {attr}')



In [ ]:
adv_acq_proxy.dwell_time

In [ ]:
adv_acq_proxy.scan_region

In [ ]:
adv_acq_proxy.scan_region = [0, 0, 1, 1]

In [ ]:
# One simultaneous acquisition
response = mic_proxy.get_images(["HAADF"])
info = json.loads(response)


In [ ]:
info

In [ ]:
import matplotlib.pyplot as plt

# Quick retrieval
images = []
for img_meta in info["images"]:
    meta_json, img_bytes = mic_proxy.get_image_data_cached(img_meta["index"])
    img = np.frombuffer(img_bytes, dtype=img_meta["dtype"]).reshape(img_meta["shape"])
    images.append((img_meta["detector"], img))

# Plot all images
fig, axes = plt.subplots(1, len(images), figsize=(6*len(images), 5))

# Handle single image case
if len(images) == 1:
    axes = [axes]

for ax, (detector_name, img) in zip(axes, images):
    im = ax.imshow(img, cmap='gray')
    ax.set_title(f"{detector_name.upper()} Detector")
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

### Acquire a cropped rectangle scan


In [ ]:
adv_acq_proxy.scan_region = [0, 0, 0.2, 0.8]
# One simultaneous acquisition
response = mic_proxy.get_images(["HAADF"])
info = json.loads(response)


In [ ]:
import matplotlib.pyplot as plt

# Quick retrieval
images = []
for img_meta in info["images"]:
    meta_json, img_bytes = mic_proxy.get_image_data_cached(img_meta["index"])
    img = np.frombuffer(img_bytes, dtype=img_meta["dtype"]).reshape(img_meta["shape"])
    images.append((img_meta["detector"], img))

# Plot all images
fig, axes = plt.subplots(1, len(images), figsize=(6*len(images), 5))

# Handle single image case
if len(images) == 1:
    axes = [axes]

for ax, (detector_name, img) in zip(axes, images):
    im = ax.imshow(img, cmap='gray')
    ax.set_title(f"{detector_name.upper()} Detector")
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.show()

### Convert acquired images to sidpy datasets


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import sidpy

# Step 1: Acquire images simultaneously
response = mic_proxy.get_images(["HAADF"])
info = json.loads(response)

# Step 2: Create sidpy datasets for each image
datasets = []

for img_meta in info["images"]:
    # Get the cached image
    meta_json, img_bytes = mic_proxy.get_image_data_cached(img_meta["index"])
    img = np.frombuffer(img_bytes, dtype=img_meta["dtype"]).reshape(img_meta["shape"])
    
    # Create sidpy dataset
    dataset = sidpy.Dataset.from_array(img, name=img_meta["detector"])
    
    # Set dimensions (assuming 0.02 nm/pixel - adjust as needed)
    dataset.set_dimension(0, sidpy.Dimension(np.arange(img.shape[0]) * 0.02, 'x'))
    dataset.set_dimension(1, sidpy.Dimension(np.arange(img.shape[1]) * 0.02, 'y'))
    
    # Set dataset metadata
    dataset.data_type = 'image'
    dataset.units = 'counts'
    dataset.quantity = 'intensity'
    dataset.title = img_meta["detector"].upper()
    
    # X dimension metadata
    dataset.x.dimension_type = 'spatial'
    dataset.x.units = 'nm'
    dataset.x.quantity = 'distance'
    
    # Y dimension metadata
    dataset.y.dimension_type = 'spatial'
    dataset.y.units = 'nm'
    dataset.y.quantity = 'distance'
    
    # Store acquisition metadata
    dataset.metadata = img_meta
    
    datasets.append(dataset)



In [ ]:
datasets

In [ ]:
view = datasets[0].plot()